# UrbanHub : flux batch météo

Sources → MinIO Bronze (CSV originaux) → Silver (Parquet nettoyé) → Gold (agrégats) → graphiques.
Le pipeline Docker assure la persistance complète. Ce notebook explique le nettoyage et les analyses, ou relit le lake via DuckDB en mémoire.


## Préparer les outils
`requests`, `pandas`, `numpy`, `pyarrow`, `matplotlib`, `seaborn` et le package `urbanhub_common` (installé dans Docker). Aucun serveur de base de données.


In [ ]:
%pip install requests pandas numpy pyarrow duckdb==1.4.3 matplotlib seaborn -q


### Les réglages
Tout ce que tu peux vouloir changer est ici, en haut, au même endroit :

| Réglage | Rôle |
|---|---|
| `ANNEES` | Les années à télécharger (par défaut : les 5 dernières) |
| `NB_THREADS` | Nombre de téléchargements **en même temps** (baisse à 5 si ça bloque) |
| `SAUVER_HORAIRE` | `True` = on envoie aussi les données heure par heure dans MinIO. Mets `False` si ta base gratuite est trop petite (limite volume disponible) |
| `MIN_OBS_JOUR` | Un jour n'est gardé que s'il a au moins ce nombre de mesures (évite les jours "pauvres") |

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path
import io

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- RÉGLAGES ----------
ANNEE_FIN = date.today().year
ANNEES = list(range(ANNEE_FIN - 4, ANNEE_FIN + 1))   # les 5 dernières années
NB_THREADS = 10          # téléchargements en même temps
SAUVER_HORAIRE = True  # conserver les observations dans MinIO Silver
ECRIRE_LAKE = False  # le pipeline Docker assure déjà la persistance complète
MIN_OBS_JOUR = 8         # nombre minimum de mesures pour garder un jour

# ---------- DOSSIERS ET ADRESSES ----------
DOSSIER_RAW = Path("data/raw")       # données brutes = notre "Data Lake"
DOSSIER_RAW.mkdir(parents=True, exist_ok=True)

URL_LISTE_STATIONS = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
URL_FICHIER = "https://www.ncei.noaa.gov/data/global-hourly/access/{annee}/{station}.csv"
# Adresse de secours : exactement les mêmes données, hébergées chez Amazon
URL_SECOURS = "https://noaa-global-hourly-pds.s3.amazonaws.com/{annee}/{station}.csv"

print("Années à télécharger :", ANNEES)

## Étape 1 – Trouver les stations météo de la France

Le site NOAA contient des stations du **monde entier**. Nous, on veut seulement la France.

La NOAA publie un fichier `isd-history.csv` qui liste **toutes** les stations, avec leur pays (colonne `CTRY`). On garde celles où `CTRY = "FR"`.

Le nom d'un fichier de données = code `USAF` (6 chiffres) + code `WBAN` (5 chiffres).
Exemple : USAF `071490` + WBAN `99999` → fichier `07149099999.csv`.

In [ ]:
def charger_stations_france():
    # 1) télécharger la liste de toutes les stations du monde
    reponse = requests.get(URL_LISTE_STATIONS, timeout=60)
    reponse.raise_for_status()
    stations = pd.read_csv(io.StringIO(reponse.text), dtype=str)

    # 2) garder seulement la France
    stations = stations[stations["CTRY"] == "FR"].copy()

    # 3) fabriquer le nom du fichier : USAF (6 chiffres) + WBAN (5 chiffres)
    stations["station_id"] = stations["USAF"].str.zfill(6) + stations["WBAN"].str.zfill(5)

    # 4) dates de début et de fin d'activité (pour éviter de demander des années inutiles)
    stations["debut"] = pd.to_datetime(stations["BEGIN"], format="%Y%m%d", errors="coerce")
    stations["fin"] = pd.to_datetime(stations["END"], format="%Y%m%d", errors="coerce")

    return stations[["station_id", "STATION NAME", "LAT", "LON", "debut", "fin"]]


stations = charger_stations_france()
print(len(stations), "stations trouvées en France")
stations.head()

## Étape 2 – Télécharger les fichiers EN PARALLÈLE

### 2a. Faire la liste du travail
Il y a un fichier par **station** et par **année**. On crée la liste de tous les couples `(station, année)`.
On saute les années où la station n'était pas en service.

In [ ]:
def construire_liste_taches(stations):
    taches = []
    for _, st in stations.iterrows():
        for annee in ANNEES:
            if pd.notna(st["debut"]) and st["debut"].year > annee:
                continue     # la station n'existait pas encore
            if pd.notna(st["fin"]) and st["fin"].year < annee:
                continue     # la station était déjà fermée
            taches.append((st["station_id"], annee))
    return taches


taches = construire_liste_taches(stations)
print(len(taches), "fichiers à essayer de récupérer")

### 2b. Une règle importante : détecter les faux fichiers
Quand un fichier **n'existe pas**, le site NOAA ne répond pas "introuvable". Il répond avec une **page web** (HTML) en disant "tout va bien". Si on ne fait pas attention, on enregistre cette page web comme si c'était des données.

La fonction `est_un_vrai_csv` regarde le début du contenu : si c'est une page web, c'est un faux fichier.

In [ ]:
def est_un_vrai_csv(contenu):
    # contenu = les octets reçus. Une page web commence par <html ou <!doctype
    debut = contenu[:200].lower()
    return len(debut) > 0 and b"<html" not in debut and b"<!doctype" not in debut


# Nettoyage : si tu as déjà téléchargé des fichiers avant, on supprime les faux
nb_supprimes = 0
for f in DOSSIER_RAW.glob("*/*.csv"):
    with open(f, "rb") as fichier:
        if not est_un_vrai_csv(fichier.read(200)):
            f.unlink()
            nb_supprimes += 1
print(nb_supprimes, "faux fichiers supprimés")

### 2c. La fonction qui télécharge UN fichier
- Si le fichier est déjà là → on ne le retélécharge pas (tu peux relancer sans tout perdre).
- On essaie l'adresse NOAA, puis l'adresse de secours.
- Si c'est un faux fichier → on n'enregistre rien (`absent`).
- Si le réseau a un problème → on réessaie 3 fois.

Elle renvoie un petit mot : `ok`, `deja`, `absent` ou `erreur`.

In [ ]:
def telecharger_un_fichier(station_id, annee):
    dossier = DOSSIER_RAW / str(annee)
    dossier.mkdir(parents=True, exist_ok=True)
    chemin = dossier / f"{station_id}.csv"

    if chemin.exists() and chemin.stat().st_size > 0:
        return "deja"

    probleme_reseau = False
    for modele in (URL_FICHIER, URL_SECOURS):          # 1) NOAA, 2) adresse de secours
        url = modele.format(annee=annee, station=station_id)
        for essai in range(3):
            try:
                reponse = requests.get(url, timeout=60)
            except requests.RequestException:
                probleme_reseau = True
                continue                                # on réessaie
            probleme_reseau = False
            if reponse.status_code == 200 and est_un_vrai_csv(reponse.content):
                chemin.write_bytes(reponse.content)
                return "ok"
            if reponse.status_code >= 500:              # le serveur a un souci : on réessaie
                probleme_reseau = True
                continue
            break                                       # le fichier n'existe pas ici : adresse suivante
    return "erreur" if probleme_reseau else "absent"

### 2d. Lancer le téléchargement en parallèle
`ThreadPoolExecutor` crée une **équipe de `NB_THREADS` ouvriers**. Chaque ouvrier prend un fichier dans la liste, le télécharge, puis en prend un autre.
Résultat : 10 fichiers arrivent en même temps, au lieu d'un seul. C'est ce que demande le sujet.

⏳ Cette cellule peut prendre plusieurs minutes.

In [ ]:
bilan = {"ok": 0, "deja": 0, "absent": 0, "erreur": 0}

with ThreadPoolExecutor(max_workers=NB_THREADS) as equipe:
    travaux = [equipe.submit(telecharger_un_fichier, s, a) for s, a in taches]
    for i, travail in enumerate(as_completed(travaux), start=1):
        bilan[travail.result()] += 1
        if i % 100 == 0 or i == len(travaux):
            print(f"{i}/{len(travaux)} traités")

print("\nBilan :", bilan)
print("ok = téléchargés | deja = déjà présents | absent = n'existent pas sur le site | erreur = problème réseau")

### 2e. Vérifier ce qu'on a reçu
Combien de **vrais** fichiers par année ?

In [ ]:
fichiers = sorted(DOSSIER_RAW.glob("*/*.csv"))
print("Total :", len(fichiers), "fichiers")
for annee in ANNEES:
    print(annee, ":", len(list((DOSSIER_RAW / str(annee)).glob("*.csv"))), "fichiers")

assert len(fichiers) > 0, "Aucun fichier valide : regarde le bilan ci-dessus (erreur = réseau)."

## Étape 3 – Lire et nettoyer les données

### Comment sont écrits les fichiers NOAA
Une case contient **plusieurs valeurs collées**, séparées par des virgules. Le dernier chiffre est un **code de qualité**.

| Colonne NOAA | Exemple | Ce que ça veut dire |
|---|---|---|
| `TMP` | `+0123,1` | température **12,3 °C** (valeur en dixièmes), qualité 1 |
| `WND` | `180,1,N,0050,1` | vent venant de **180°**, vitesse **5,0 m/s** (en dixièmes) |
| `SLP` | `10132,1` | pression **1013,2 hPa** (en dixièmes) |
| `VIS` | `010000,1,N,1` | visibilité **10 000 m** |
| `AA1` | `01,0020,3,1` | pluie : mesure sur **1 h**, **2,0 mm** (en dixièmes) |

### Les nettoyages qu'on fait
1. **Couper** chaque case pour récupérer les valeurs.
2. **Valeurs manquantes** : `9999`, `99999`, `999999`... veulent dire "pas de mesure" → on met vide (`NaN`).
3. **Qualité** : si le code qualité dit "douteux" ou "faux" (2, 3, 6, 7) → on met vide.
4. **Unités** : on divise par 10 quand c'est en dixièmes, et on passe le vent de m/s en **km/h**.
5. **Timestamp** : le texte devient une vraie date/heure (UTC), arrondie à l'heure.
6. **Valeurs impossibles** (ex : 80 °C) → vide.
7. **Doublons** : une seule mesure par station et par heure.

In [ ]:
COLONNES = ["STATION", "DATE", "WND", "VIS", "TMP", "SLP", "AA1"]
MAUVAISE_QUALITE = ["2", "3", "6", "7"]      # codes NOAA : mesure douteuse ou fausse

# Valeurs physiquement possibles. En dehors = erreur de capteur.
LIMITES = {
    "temperature":    (-50, 55),      # °C
    "wind_speed":     (0, 300),       # km/h
    "wind_direction": (0, 360),       # degrés
    "pressure":       (850, 1090),    # hPa
    "visibility":     (0, 100000),    # mètres
    "precipitation":  (0, 300),       # mm
}
MESURES = list(LIMITES)


def decouper(serie, nb):
    # Coupe '180,1,N,0050,1' en colonnes : ['180', '1', 'N', '0050', '1']
    morceaux = serie.fillna("").astype(str).str.split(",", expand=True)
    return morceaux.reindex(columns=range(nb))


def en_nombre(morceaux, i_val, i_qual, manquant, echelle):
    # Prend une valeur, enlève 'manquant' et les mauvaises qualités, puis change l'échelle
    val = pd.to_numeric(morceaux[i_val], errors="coerce")
    val = val.where(val != manquant)                          # 9999 -> vide
    val = val.where(~morceaux[i_qual].isin(MAUVAISE_QUALITE)) # mesure douteuse -> vide
    return val / echelle                                      # ex : 123 -> 12.3


def lire_et_nettoyer(chemin):
    df = pd.read_csv(chemin, usecols=lambda c: c in COLONNES, dtype=str,
                     on_bad_lines="skip").reindex(columns=COLONNES)

    wnd = decouper(df["WND"], 5)   # vent : direction, qualité, type, vitesse, qualité
    vis = decouper(df["VIS"], 4)   # visibilité
    tmp = decouper(df["TMP"], 2)   # température
    slp = decouper(df["SLP"], 2)   # pression
    aa1 = decouper(df["AA1"], 4)   # pluie : durée (h), quantité, condition, qualité

    out = pd.DataFrame({
        "station_id":      df["STATION"],
        "timestamp":       pd.to_datetime(df["DATE"], utc=True, errors="coerce"),
        "wind_speed":      en_nombre(wnd, 3, 4, 9999, 10) * 3.6,   # dixièmes de m/s -> km/h
        "pressure":        en_nombre(slp, 0, 1, 99999, 10),         # dixièmes de hPa -> hPa
        "wind_direction":  en_nombre(wnd, 0, 1, 999, 1),            # degrés
        "precipitation":   en_nombre(aa1, 1, 3, 9999, 10),          # dixièmes de mm -> mm
        "precip_period_h": en_nombre(aa1, 0, 3, 99, 1),             # durée de la mesure de pluie
        "temperature":     en_nombre(tmp, 0, 1, 9999, 10),          # dixièmes de °C -> °C
        "visibility":      en_nombre(vis, 0, 1, 999999, 1),         # mètres
    })

    for col, (mini, maxi) in LIMITES.items():                       # valeurs impossibles -> vide
        out[col] = out[col].where(out[col].between(mini, maxi))

    out = out.dropna(subset=["timestamp"])
    out["timestamp"] = out["timestamp"].dt.floor("h")               # arrondi à l'heure
    return out.drop_duplicates(["station_id", "timestamp"])         # 1 mesure / station / heure

### 3b. Test sur UN fichier
Avant de traiter les ~800 fichiers, on regarde le résultat sur un seul. Compare avec le fichier brut pour vérifier que les valeurs sont logiques (ex : température entre -20 et 40).

In [ ]:
f = fichiers[0]
print("Fichier :", f)

brut = pd.read_csv(f, dtype=str, on_bad_lines="skip")
print("\n--- BRUT (extrait) ---")
display(brut[[c for c in ["STATION", "DATE", "WND", "VIS", "TMP", "SLP", "AA1"] if c in brut.columns]].head(3))

propre = lire_et_nettoyer(f)
print("--- PROPRE ---")
print(len(brut), "lignes brutes ->", len(propre), "lignes propres")
display(propre.head(3))

### 3c. Nettoyer TOUS les fichiers et les assembler en un seul tableau
On lit chaque fichier, on le nettoie, puis on colle tout ensemble dans le tableau `meteo`.
On en garde aussi une copie dans `data/weather_france_clean.parquet` (un format rapide et compact).

In [ ]:
morceaux = []
for i, f in enumerate(fichiers, start=1):
    morceaux.append(lire_et_nettoyer(f))
    if i % 100 == 0 or i == len(fichiers):
        print(f"{i}/{len(fichiers)} fichiers nettoyés")

meteo = pd.concat(morceaux, ignore_index=True)
meteo = meteo.dropna(subset=MESURES, how="all")                    # lignes sans aucune mesure -> supprimées
meteo = meteo.drop_duplicates(["station_id", "timestamp"])
meteo = meteo.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

Path("data").mkdir(exist_ok=True)
meteo.to_parquet("data/weather_france_clean.parquet")

print("\n", len(meteo), "lignes |", meteo["station_id"].nunique(), "stations |",
      meteo["timestamp"].min().date(), "->", meteo["timestamp"].max().date())

### 3d. Contrôle qualité
- Le **% de valeurs manquantes** par colonne (normal : la pluie en a beaucoup, car elle n'est pas mesurée partout).
- Le **résumé statistique** : regarde les minimum / maximum, ils doivent être réalistes.

In [ ]:
print("% de valeurs manquantes :")
print((meteo.isna().mean() * 100).round(1))
meteo[MESURES].describe().round(1)

## Étape 4 : agrégats et fichiers du lake
Le pipeline Docker archive les CSV originaux puis écrit les Parquet Silver/Gold. Les cellules suivantes permettent aussi de publier les tableaux déjà calculés, avec `ECRIRE_LAKE=True`.


In [ ]:
# Un tableau temporaire avec la date du jour (en heure de Paris) et la pluie sur 1 heure
temp = meteo.assign(
    date=meteo["timestamp"].dt.tz_convert("Europe/Paris").dt.date,
    pluie_1h=meteo["precipitation"].where(meteo["precip_period_h"] == 1),
)

daily = (temp.groupby(["station_id", "date"])
         .agg(temp_mean=("temperature", "mean"),
              temp_min=("temperature", "min"),
              temp_max=("temperature", "max"),
              wind_max=("wind_speed", "max"),
              pressure_mean=("pressure", "mean"),
              visibility_mean=("visibility", "mean"),
              precipitation=("pluie_1h", "sum"),
              n_pluie=("pluie_1h", "count"),
              nb_obs=("temperature", "size"))
         .reset_index())
daily.loc[daily["n_pluie"] == 0, "precipitation"] = np.nan     # pas de mesure de pluie ce jour-là
daily = daily.drop(columns="n_pluie")
daily = daily[daily["nb_obs"] >= MIN_OBS_JOUR].reset_index(drop=True)   # on garde les jours bien renseignés
del temp

# Liste des stations qui ont des données
liste_stations = (stations[["station_id", "STATION NAME", "LAT", "LON"]]
                  .rename(columns={"STATION NAME": "name", "LAT": "lat", "LON": "lon"}))
liste_stations["lat"] = pd.to_numeric(liste_stations["lat"], errors="coerce")
liste_stations["lon"] = pd.to_numeric(liste_stations["lon"], errors="coerce")
liste_stations = liste_stations[liste_stations["station_id"].isin(meteo["station_id"].unique())]

print(len(daily), "lignes 'par jour' |", len(liste_stations), "stations")
daily.head()

### Préparer le client MinIO
Les variables `S3_*` viennent de `.env`. Aucun lien de connexion SQL ni mot de passe de base ne sont demandés.


In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")
from urbanhub_common import write_parquet, read_sql


### Publier les fichiers Parquet
Les objets ajoutés restent immuables. Les vues dédupliquent les clés métier à la lecture. Pour une ingestion complète avec Bronze, lancer le pipeline `batch-meteo`.


In [ ]:
if ECRIRE_LAKE:
    write_parquet(daily.assign(station_id=daily.station_id.astype(str)), "gold", "meteo/daily", "date")
    if SAUVER_HORAIRE:
        write_parquet(meteo, "silver", "noaa/global-hourly", "timestamp")
else:
    print("Publication ignorée : lancer le pipeline Docker pour l'ingestion complète.")


### Vérifier dans MinIO
Ouvrir http://localhost:9001 puis le bucket `urbanhub`. Les données sont des objets sous Bronze/Silver/Gold.


In [ ]:
if ECRIRE_LAKE:
    print(read_sql("SELECT count(*) AS lignes FROM meteo.daily"))


## Charger les données déjà dans le lake
Si les tableaux ne sont pas en mémoire, lire les Parquet MinIO avec les vues DuckDB temporaires.


In [ ]:
if "daily" not in globals():
    from dotenv import load_dotenv
    load_dotenv("../.env")
    from urbanhub_common import read_sql
    daily = read_sql("SELECT * FROM meteo.daily")
    liste_stations = read_sql("SELECT station_id,name,latitude AS lat,longitude AS lon FROM meteo.stations")
    print(len(daily), "jours", len(liste_stations), "stations depuis MinIO")
else:
    print("Données déjà en mémoire.")


## Étape 5 – Répondre aux 4 questions métier

On travaille avec le tableau `daily` (un résumé par station et par jour).

### Préparation : associer chaque station à une grande ville
Pour la question 3, on veut la température **par ville**. On rattache chaque station à la ville la plus proche, si elle est à moins de 30 km. La distance est calculée avec les coordonnées GPS.

In [ ]:
daily["date"] = pd.to_datetime(daily["date"])

VILLES = {"Paris": (48.85, 2.35), "Lyon": (45.76, 4.84), "Marseille": (43.30, 5.37),
          "Lille": (50.63, 3.06), "Toulouse": (43.60, 1.44), "Bordeaux": (44.84, -0.58),
          "Nantes": (47.22, -1.55), "Strasbourg": (48.57, 7.75), "Nice": (43.70, 7.27)}


def distance_km(lat1, lon1, lat2, lon2):
    # distance entre deux points GPS (formule de "haversine")
    p = np.pi / 180
    a = (0.5 - np.cos((lat2 - lat1) * p) / 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2)
    return 12742 * np.arcsin(np.sqrt(a))


st = liste_stations.dropna(subset=["lat", "lon"]).copy()
for nom, (la, lo) in VILLES.items():
    st[nom] = distance_km(st["lat"], st["lon"], la, lo)
st["ville"] = st[list(VILLES)].idxmin(axis=1)
st.loc[st[list(VILLES)].min(axis=1) > 30, "ville"] = np.nan        # trop loin = pas de ville

daily["ville"] = daily["station_id"].map(st.set_index("station_id")["ville"])
print("Nombre de jours de mesure par ville :")
print(daily["ville"].value_counts())

### Question 1 – Peut-on identifier des périodes météorologiques anormales ?

**Idée :** pour chaque jour, on compare la température de la France à la température **habituelle de ce mois**.
Si l'écart est très grand (plus de **2 écarts-types**, c'est ce qu'on appelle un *z-score*), le jour est **anormal**.
Ensuite, on regroupe les jours anormaux **qui se suivent** en périodes (par exemple une canicule ou une vague de froid).

In [ ]:
france = daily.groupby("date")["temp_mean"].mean()          # température moyenne France, jour par jour
mois = france.index.month

moyenne_mois = france.groupby(mois).transform("mean")
ecart_type_mois = france.groupby(mois).transform("std")
z = (france - moyenne_mois) / ecart_type_mois               # écart à la normale du mois

anormaux = france[z.abs() > 2]
print(len(anormaux), "jours anormaux sur", len(france))

# regrouper les jours anormaux consécutifs en périodes
groupe = (anormaux.index.to_series().diff().dt.days != 1).cumsum()
periodes = anormaux.groupby(groupe).agg(debut=lambda s: s.index.min(),
                                         fin=lambda s: s.index.max(),
                                         nb_jours="size",
                                         temperature_moyenne="mean")
periodes["temperature_moyenne"] = periodes["temperature_moyenne"].round(1)
display(periodes.sort_values("nb_jours", ascending=False).head(10))

plt.figure(figsize=(12, 4))
plt.plot(france, lw=0.8, label="Température moyenne France")
plt.scatter(anormaux.index, anormaux, color="red", s=14, label="Jours anormaux")
plt.title("Température moyenne en France et jours anormaux")
plt.ylabel("°C"); plt.legend(); plt.show()

> 📖 **Comment lire ce graphique.** La courbe bleue donne la température moyenne de la France jour après jour. Les points rouges sont les jours **anormaux** (|z| > 2), c'est-à-dire plus de 2 écarts-types au-dessus ou au-dessous de la normale de leur mois. Le tableau regroupe les jours anormaux consécutifs en **périodes**, triées par durée.

> ⚠️ **Attention au biais 2026.** NOAA s'arrête au 24/08/2025 ; ensuite, `daily` ne contient que les **11 points
> Open-Meteo** des grandes villes. La « moyenne France » change donc de composition : en 2022-2024, la moyenne de
> toutes les stations est **0,74 °C plus froide** que celle des 11 villes (stations rurales et de montagne).
> 2026 paraît donc artificiellement chaud : 30 jours « chauds » avec cette méthode. La cellule suivante refait le
> calcul sur une **série homogène** (moyenne des 9 villes du notebook sur toute la période) : il reste 23 jours chauds en 2026 (21 avec les 11 villes de `ref.cities`, comme sur le dashboard).
> Ces jours restent un peu surestimés, car le modèle Open-Meteo est 0,3 °C plus chaud que NOAA (vérifié sur 2024).

In [ ]:
# Série homogène : moyenne des villes (stations NOAA < 30 km jusqu'en 2025, puis points Open-Meteo)
if "ville" not in daily.columns:   # même rattachement que la cellule « Préparation »
    st_v = liste_stations.dropna(subset=["lat", "lon"]).copy()
    for nom, (la, lo) in VILLES.items():
        st_v[nom] = distance_km(st_v["lat"], st_v["lon"], la, lo)
    st_v["ville"] = st_v[list(VILLES)].idxmin(axis=1)
    st_v.loc[st_v[list(VILLES)].min(axis=1) > 30, "ville"] = np.nan
    daily["ville"] = daily["station_id"].map(st_v.set_index("station_id")["ville"])
villes_j = daily.dropna(subset=["ville"]).groupby(["date", "ville"])["temp_mean"].mean().groupby("date").mean()
zv = (villes_j - villes_j.groupby(villes_j.index.month).transform("mean")) / villes_j.groupby(villes_j.index.month).transform("std")
an_v = zv[zv.abs() > 2]
print(len(an_v), "jours anormaux (série homogène) ;", "par année et par signe :")
pd.DataFrame({"annee": an_v.index.year, "type": np.where(an_v > 0, "chaud", "froid")}).value_counts().unstack(fill_value=0)


**Comment répondre :** *Oui.* Cite 2 ou 3 périodes du tableau (dates, nombre de jours, température) et dis si ce sont des périodes chaudes (canicule) ou froides.

**Résultats obtenus (données UrbanHub, 2022 → octobre 2026).** **68 jours anormaux sur 1 751** (environ 4 %), répartis en
périodes. Les plus longues sont **chaudes** : 23-30 mai 2026 (8 jours, 25,6 °C, z ≈ 2,9), 21-27 juin 2026 (7 jours,
30 °C : canicule), 9-14 juillet 2026, 12-14 août 2026. Les **froides** sont plus courtes : 1-4 avril 2022 (4,4 °C),
27 février-2 mars 2023 (3,2 °C). Les vagues de chaleur de 2026 restent nettes sur la série homogène, mais leur
*nombre* est gonflé par le changement de source (voir l'encadré ⚠️).

### Question 2 – Existe-t-il une corrélation entre les conditions météo et la visibilité ?

**Idée :** la corrélation est un nombre entre **-1 et +1** :
- proche de **+1** : quand l'un monte, l'autre monte ;
- proche de **-1** : quand l'un monte, l'autre baisse ;
- proche de **0** : pas de lien.

On utilise la méthode *Spearman*, mieux adaptée quand les données ne sont pas "régulières" (ce qui est le cas de la visibilité).

In [ ]:
colonnes = ["visibility_mean", "temp_mean", "wind_max", "pressure_mean", "precipitation"]
corr = daily[colonnes].corr(method="spearman")

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Corrélation entre la météo et la visibilité"); plt.show()

print("Corrélation avec la visibilité :")
print(corr["visibility_mean"].drop("visibility_mean").sort_values().round(2))

> 📖 **Comment lire ce graphique.** Carte de chaleur des corrélations de Spearman entre variables journalières : rouge = elles montent ensemble, bleu = l'une monte quand l'autre baisse, blanc = pas de lien. La liste du bas isole la ligne `visibility_mean`.

**Comment répondre :** regarde la ligne `visibility_mean`. Le chiffre le plus éloigné de 0 est la variable la plus liée à la visibilité. En général : la **pluie** fait baisser la visibilité (corrélation négative). Attention : une corrélation ne prouve pas que l'une **cause** l'autre.

**Résultats obtenus.** Les corrélations avec la visibilité sont **faibles** : pluie **-0,16**, pression -0,10, vent
0,02, température **+0,20**. La pluie et l'air stable des anticyclones (brume, particules piégées) réduisent la
visibilité, alors que l'air chaud et sec l'améliore. À l'échelle de la journée, la visibilité dépend surtout
d'événements brefs (brouillard matinal, averses) que la moyenne journalière lisse : il faudrait travailler en
**horaire** (données silver) pour aller plus loin.

### Question 3 – Quelle est l'évolution saisonnière de la température (France et grandes villes) ?

**Idée :** on calcule la température moyenne de **chaque mois** (janvier à décembre), pour la France entière puis pour chaque ville.

In [ ]:
daily["mois"] = daily["date"].dt.month

france_mois = daily.groupby("mois")["temp_mean"].mean()
villes_mois = (daily.dropna(subset=["ville"])
               .groupby(["ville", "mois"])["temp_mean"].mean().unstack(0))

plt.figure(figsize=(10, 5))
plt.plot(france_mois, color="black", lw=3, label="France")
for v in villes_mois.columns:
    plt.plot(villes_mois[v], lw=1.3, label=v)
plt.xticks(range(1, 13)); plt.xlabel("Mois"); plt.ylabel("Température moyenne (°C)")
plt.title("Cycle saisonnier de la température"); plt.legend(ncol=2); plt.show()

display(villes_mois.round(1))

> 📖 **Comment lire ce graphique.** Courbe noire = France, courbes colorées = villes. On compare la **forme** (minimum, maximum) et le **niveau** (écart entre villes). Le tableau donne la valeur exacte par mois et par ville.

**Comment répondre :** la température suit une courbe en "cloche" : minimum en **janvier**, maximum en **juillet-août**. Compare les villes : le Sud (Marseille, Nice) est plus chaud que le Nord (Lille, Strasbourg).

**Résultats obtenus.** France : **5,8 °C en janvier**, **22,0 °C en août** (maximum en juillet-août). Le **Sud** est le plus
chaud : Nice 9,6 °C en janvier et 26,5 °C en juillet, Marseille 8,0 et 27,0 °C. Le **Nord-Est** est le plus froid en
hiver : Strasbourg 3,3 °C, Lille 4,3 °C. L'amplitude janvier-juillet est plus forte dans l'Est continental (Lyon et
Strasbourg : environ 19 °C) que dans l'Ouest océanique (Rennes : 14 °C, Nantes : 15 °C), car l'océan tempère les saisons.

### Question 4 – Quels sont les jours présentant des conditions météo extrêmes ?

**Idée :** on calcule des valeurs moyennes **pour la France** chaque jour, puis on prend le **top 5** de chaque catégorie : les plus chauds, les plus froids, les plus venteux, les plus pluvieux.

In [ ]:
jours = daily.groupby("date").agg(temp_max=("temp_max", "mean"),
                                  temp_min=("temp_min", "mean"),
                                  vent_max=("wind_max", "mean"),
                                  pluie=("precipitation", "mean"))

print("Les 5 jours les plus chauds (température max moyenne, °C) :")
display(jours["temp_max"].nlargest(5).round(1))
print("Les 5 jours les plus froids (température min moyenne, °C) :")
display(jours["temp_min"].nsmallest(5).round(1))
print("Les 5 jours les plus venteux (vent max moyen, km/h) :")
display(jours["vent_max"].nlargest(5).round(1))
print("Les 5 jours les plus pluvieux (pluie moyenne, mm) :")
display(jours["pluie"].nlargest(5).round(1))

> 📖 **Comment lire ce graphique.** Top 5 des jours pour chaque catégorie, à partir de la moyenne nationale des maximums, minimums, rafales et cumuls de pluie. Une valeur nationale élevée signifie que l'événement a touché beaucoup de stations en même temps.

**Résultats obtenus (Q4).** Jours les plus **chauds** : 22-25 juin 2026 (maximum moyen d'environ 37,5 °C, canicule) et
13 août 2026. Les plus **froids** : 14-15 janvier 2025 (minimum moyen de -4,6 °C) et 20 janvier 2024. Les plus
**venteux** : **2 novembre 2023, tempête Ciaran** (52 km/h de vent max moyen) et 10 mars 2023. Les plus **pluvieux** :
21 septembre 2025 (15,5 mm en moyenne) et 20 octobre 2026. ⚠️ Les dates de 2026 viennent d'Open-Meteo (11 villes),
celles d'avant de NOAA : les valeurs ne sont pas strictement comparables.

## Conclusion
Téléchargement parallèle → fichiers originaux MinIO Bronze → nettoyage → Parquet Silver → agrégats Gold → analyses. DuckDB sert uniquement à interroger les fichiers en mémoire. La conservation des données brutes permet de reprendre le nettoyage.
